# Evaluación del Sistema RAG (LLM-as-a-judge)
Este cuaderno implementa la evaluación automatizada del sistema RAG de Ingeniería de Requerimientos utilizando el enfoque de **LLM-as-a-judge**.

## ¿Por qué utilizar LLM-as-a-judge?
La evaluación manual de las respuestas generadas por un sistema RAG es un proceso lento y poco escalable. A medida que el dataset de requerimientos crece (actualmente ~170, pero en aumento), requerimos un mecanismo robusto y automático para medir la calidad.

El enfoque de *LLM-as-a-judge* emplea un Modelo de Lenguaje Grande como evaluador imparcial que compara las salidas del sistema contra criterios objetivos definidos en *prompts* especializados, garantizando:
1. **Escalabilidad** sin esfuerzo humano.
2. **Evaluación Multidimensional** más allá de la similitud de palabras (BLEU/ROUGE).
3. **Alineación** con los estándares modernos de validación de sistemas RAG.

## Métricas Cuantificables Implementadas
1. **Context Relevance (Relevancia del Contexto)**: ¿Son útiles los fragmentos recuperados del PDF ISO 29148 y dataset vectorial para analizar este requerimiento en particular? (Escala 1-5).
2. **Faithfulness (Fidelidad)**: ¿La respuesta se basa en la norma ISO recuperada o alucina justificaciones inventadas? (Escala 1-5).
3. **Answer Relevance (Relevancia de la Respuesta)**: ¿La retroalimentación responde directamente a los defectos del requerimiento? (Escala 1-5).
4. **Domain Accuracy (Precisión del Dominio)**: Comparación exacta de los puntajes generados (VERIFIABILITY, ATOMICITY, etc.) contra el *Gold Standard* definido en `requirements.csv` (Exact Match).

## 1. Configuración del Entorno

In [35]:
import os
import json
import requests
import pandas as pd
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

load_dotenv('../.env')

API_URL = "http://localhost:8000"

LOCAL_MODELS_API = os.getenv("LOCAL_MODELS_API", "http://localhost:1234/v1/").replace('host.docker.internal', 'localhost')
MODEL_NAME = os.getenv("LOCAL_MODEL", "lmstudio-community/Meta-Llama-3-8B-Instruct-GGUF")

API_KEY = os.getenv("GOOGLE_API_KEY")

GEMINI_MODEL= os.getenv("GEMINI_MODEL")

print(f"RAG API Endpoint: {API_URL}")
print(f"LLM-as-a-judge API Endpoint: {LOCAL_MODELS_API}")
print(f"Model Name: {GEMINI_MODEL}")

judge_llm = ChatGoogleGenerativeAI(
    api_key=API_KEY,
    model=GEMINI_MODEL,
    temperature=0.0
)

RAG API Endpoint: http://localhost:8000
LLM-as-a-judge API Endpoint: http://localhost:1234/v1/
Model Name: models/gemini-2.5-flash


## 2. Carga y Preparación de Datos

In [ ]:
df_reqs = pd.read_csv('../data/raw/requirements.csv')
print(f"Total de requerimientos cargados: {len(df_reqs)}")

def parse_tags(tag_str):
    """Parse the true tags"""
    if pd.isna(tag_str) or not isinstance(tag_str, str):
        return {}
    tags = {}
    for pair in tag_str.split(';'):
        if ':' in pair:
            k, v = pair.split(':')
            try:
                tags[k.strip().upper()] = int(v.strip())
            except ValueError:
                pass
    return tags

df_reqs['true_tags'] = df_reqs['tags'].apply(parse_tags)

# 5 random examples 
test_dataset = df_reqs.sample(n=3, random_state=42).copy()
display(test_dataset[['requirement', 'true_tags', 'total']])

Total de requerimientos cargados: 173


,requirement,true_tags,total
162,El sistema debe contar con un módulo web para ...,"{'VERIFIABILITY': 2, 'ATOMICITY': 2, 'AMBIGUIT...",8
42,El administrador puede eliminar un servicio de...,"{'VERIFIABILITY': 2, 'ATOMICITY': 2, 'AMBIGUIT...",8
90,Los expedientes electrónicos deben crearse vin...,"{'VERIFIABILITY': 2, 'ATOMICITY': 2, 'AMBIGUIT...",9


## 3. Consultas al Sistema RAG

In [37]:
def evaluate_with_rag(requirement):
    try:
        res = requests.post(f"{API_URL}/analyze", json={"text": requirement})
        if res.status_code == 200:
            return res.json()
        return None
    except Exception as e:
        print(f"Error connecting to RAG: {e}")
        return None

results = []
for idx, row in test_dataset.iterrows():
    print(f"Analizando requerimiento {idx}...")
    rag_output = evaluate_with_rag(row['requirement'])
    
    results.append({
        "id": idx,
        "requirement": row['requirement'],
        "true_tags": row['true_tags'],
        "rag_context": "\n\n".join(rag_output.get('context_used', [])) if rag_output else "",
        "rag_evaluation": rag_output.get('evaluation', {}) if rag_output else {},
        "rag_feedback": rag_output.get('feedback', {}).get('improved_requirement', "") if rag_output else ""
    })

df_results = pd.DataFrame(results)

Analizando requerimiento 162...
Analizando requerimiento 42...
Analizando requerimiento 90...


## 4. LLM-as-a-judge: Definición de Prompts y Evaluación

In [38]:
parser = JsonOutputParser()

# context Relevance Prompt
context_relevance_prompt = PromptTemplate(
    template="""Eres un juez evaluando un sistema RAG de Ingeniería de Requerimientos.
Evalúa la "Context Relevance" (Relevancia del Contexto) en una escala del 1 al 5.
El contexto debe contener información normativa (como ISO 29148) útil para juzgar la calidad del requerimiento.

Requerimiento: {requirement}
Contexto Recuperado: {context}

Devuelve SOLAMENTE un JSON con el siguiente formato:
{{
    "score": <int entre 1 y 5>,
    "reason": "<explicación breve>"
}}
""",
    input_variables=["requirement", "context"],
)

# faithfulness prompt
faithfulness_prompt = PromptTemplate(
    template="""Eres un juez evaluando un sistema RAG de Ingeniería de Requerimientos.
Evalúa la "Faithfulness" (Fidelidad) en una escala del 1 al 5.
Determina si la evaluación generada por el sistema se basa en el contexto normativo proporcionado o si inventa (alucina) reglas.

Contexto Normativo: {context}
Evaluación Generada: {evaluation}

Devuelve SOLAMENTE un JSON con el siguiente formato:
{{
    "score": <int entre 1 y 5>,
    "reason": "<explicación breve>"
}}
""",
    input_variables=["context", "evaluation"],
)

# answer relevance prompt
answer_relevance_prompt = PromptTemplate(
    template="""Eres un juez evaluando un sistema RAG de Ingeniería de Requerimientos.
Evalúa la "Answer Relevance" (Relevancia de la Respuesta) en una escala del 1 al 5.
La retroalimentación generada debe abordar directamente los defectos del requerimiento y proponer una mejora clara, sin desviarse del tema.

Requerimiento Original: {requirement}
Feedback y Mejora Generada: {feedback}

Devuelve SOLAMENTE un JSON con el siguiente formato:
{{
    "score": <int entre 1 y 5>,
    "reason": "<explicación breve>"
}}
""",
    input_variables=["requirement", "feedback"],
)

context_chain = context_relevance_prompt | judge_llm | parser
faithfulness_chain = faithfulness_prompt | judge_llm | parser
answer_chain = answer_relevance_prompt | judge_llm | parser

In [39]:
# Ejecutar el Juez en los resultados
judgements = []

for idx, row in df_results.iterrows():
    print(f"Juzgando ID {row['id']}...")
    
    try:
        cr_res = context_chain.invoke({
            "requirement": row['requirement'],
            "context": row['rag_context']
        })
        
        f_res = faithfulness_chain.invoke({
            "context": row['rag_context'],
            "evaluation": json.dumps(row['rag_evaluation'])
        })
        
        ar_res = answer_chain.invoke({
            "requirement": row['requirement'],
            "feedback": row['rag_feedback']
        })
        
        # domain accuracy
        true_tags = row['true_tags']
        pred_tags = {}
        for k, v in row['rag_evaluation'].items():
            if isinstance(v, dict) and 'score' in v:
                pred_tags[k] = v['score']
                
        # accuracy
        matches = 0
        total_keys = len(true_tags) if true_tags else 1
        for k, v in true_tags.items():
            if k in pred_tags and pred_tags[k] == v:
                matches += 1
        
        accuracy = (matches / total_keys) * 100 if total_keys > 0 else 0
        
        judgements.append({
            "id": row['id'],
            "Context_Relevance_Score": cr_res.get('score', 0),
            "Faithfulness_Score": f_res.get('score', 0),
            "Answer_Relevance_Score": ar_res.get('score', 0),
            "Domain_Accuracy_%": accuracy,
            "Predicted_Tags": pred_tags,
            "True_Tags": true_tags
        })
    except Exception as e:
        print(f"Error evaluando con el juez: {e}")

df_judgements = pd.DataFrame(judgements)

Juzgando ID 162...
Error evaluando con el juez: Error calling model 'models/gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 19.63699104s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDa

## 5. Resultados Finales y Tabla Resumen

In [40]:
display(df_judgements[['id', 'Context_Relevance_Score', 'Faithfulness_Score', 'Answer_Relevance_Score', 'Domain_Accuracy_%']])

print("MÉTRICAS AGREGADAS (PROMEDIO):")
print(f"Context Relevance (1-5): {df_judgements['Context_Relevance_Score'].mean():.2f}")
print(f"Faithfulness (1-5): {df_judgements['Faithfulness_Score'].mean():.2f}")
print(f"Answer Relevance (1-5): {df_judgements['Answer_Relevance_Score'].mean():.2f}")
print(f"Domain Accuracy (%): {df_judgements['Domain_Accuracy_%'].mean():.2f}%")


KeyError: "None of [Index(['id', 'Context_Relevance_Score', 'Faithfulness_Score',\n       'Answer_Relevance_Score', 'Domain_Accuracy_%'],\n      dtype='str')] are in the [columns]"